# Auditoría de coherencia del PAA

Verifica el nuevo flujo de niveles de actividad FIRMS y la ausencia de conteo contemporáneo en los predictores.

In [1]:
from pathlib import Path
import pandas as pd
N=Path("data/processed/modelado/dataset_niveles_actividad_firms.parquet");M=Path("data/processed/modelado/dataset_modelado_nivel_actividad_firms.parquet")
n=pd.read_parquet(N);d=pd.read_parquet(M)
for x in (n,d):x["fecha_inicio_semana"]=pd.to_datetime(x.fecha_inicio_semana)
CLAVES=["departamento","fecha_inicio_semana"]

## 1. Integridad y target

In [2]:
for nombre,x in {"niveles":n,"modelado":d}.items():
 assert len(x)==7923 and x.departamento.nunique()==19 and not x.duplicated(CLAVES).any()
 assert x.fecha_inicio_semana.min()==pd.Timestamp("2018-01-01") and x.fecha_inicio_semana.max()==pd.Timestamp("2025-12-22")
assert set(n.nivel_actividad_firms)=={"Sin actividad","Bajo","Moderado","Alto"}
assert n.loc[n.cantidad_detecciones.eq(0),"nivel_actividad_firms"].eq("Sin actividad").all()
assert n.loc[n.cantidad_detecciones.gt(0),"nivel_actividad_firms"].ne("Sin actividad").all()
print("Integridad y target correctos")

Integridad y target correctos


## 2. Lags y leakage

In [3]:
LAGS=[f"cantidad_detecciones_lag{i}" for i in range(1,5)]
# Se comprueban desde la quinta fila objetivo; las primeras usan historia 2017 que no está en este panel.
o=d.sort_values(CLAVES)
for lag in range(1,5):
 esp=o.groupby("departamento").cantidad_detecciones.shift(lag);mask=esp.notna();assert o.loc[mask,LAGS[lag-1]].eq(esp[mask]).all()
X=["departamento","mes","temperature_2m_max_mean_weekly","relative_humidity_2m_min_mean_weekly","wind_speed_10m_max_mean_weekly","precipitation_sum_weekly"]+LAGS+["promedio_detecciones_ultimas_4_semanas","max_detecciones_ultimas_4_semanas","semanas_con_actividad_ultimas_4"]
assert {"cantidad_detecciones","nivel_actividad_firms","nivel_actividad_firms_codigo"}.isdisjoint(X) and not d[X].isna().any().any();print("Lags y exclusiones correctos")

Lags y exclusiones correctos


## 3. Estado

In [4]:
pd.DataFrame([
["Calidad","Vigente","Unidad, cobertura, ceros y asimetría"],["K-Means","Vigente","Sólo log1p del conteo positivo"],["Ceros","Vigente","Sin actividad explícita"],["Target","Vigente","Cuatro niveles FIRMS"],["Modelos","Vigente","Dummy, Logística y Random Forest"],["Evaluación","Vigente","Aleatoria estratificada y temporal 2025"],["Nesterov","Auxiliar","Fuera del clustering y modelo"],["MIRA/FIREDpy","Exploratorio","Fuera del pipeline"],["Clustering meteorológico/perfiles","Obsoleto","Reemplazado"],["presencia_firms y M1–M4","Obsoleto","Reemplazado"]],columns=["Componente","Estado","Evidencia"])

,Componente,Estado,Evidencia
0,Calidad,Vigente,"Unidad, cobertura, ceros y asimetría"
1,K-Means,Vigente,Sólo log1p del conteo positivo
2,Ceros,Vigente,Sin actividad explícita
3,Target,Vigente,Cuatro niveles FIRMS
4,Modelos,Vigente,"Dummy, Logística y Random Forest"
5,Evaluación,Vigente,Aleatoria estratificada y temporal 2025
6,Nesterov,Auxiliar,Fuera del clustering y modelo
7,MIRA/FIREDpy,Exploratorio,Fuera del pipeline
8,Clustering meteorológico/perfiles,Obsoleto,Reemplazado
9,presencia_firms y M1–M4,Obsoleto,Reemplazado


## Conclusión

Flujo vigente: FIRMS histórico → log1p → K-Means → niveles de actividad → clasificación con meteorología, contexto y antecedentes. Actividad FIRMS no equivale a riesgo ni incendio confirmado.